# 01 家事と経済的負担

**F4：パートナーシップの安定性と意思決定**（家事分担・生活費の負担のモデル）

- 2 人（Partner A・B）の**所得・労働時間・家事に使える時間**を確かめる
- いろいろな分担ルールで、**家事時間**と**生活費の負担額**を計算する
- その分担が**実行可能**か（家事に使える時間・所得を超えないか）を調べる
- 計算：`partners.py`・`household_model.py`

> **教育用の架空データ・架空モデル**です。Partner A・B は実在の人物ではなく、性別や役割を意味しません。満足度関数や関係解消確率の係数は架空の値で、実際の離婚率や特定の関係の将来を予測するものではありません。

## 準備：実験環境（最初に必ず実行）

F4 リポジトリの `.py`（計算の本体）を import できるように、作業ディレクトリ（F4 のルート）を自動で探して設定します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter / VSCode | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのリポジトリの URL** を入れる |
| Colab（方法B：ZIP をアップロード） | F4 フォルダを ZIP にして `/content` にアップロードしておく（またはセルの実行中にアップロード） |

Colab の CPU ランタイムで動きます（GPU・TPU は不要）。`textbook/`（教科書のコード）は公開リポジトリに含めない約束です。`textbook/` がないと import が失敗します。

In [ ]:
#@title 準備（F4 の .py を読み込む）
import os, sys, glob, zipfile, subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                                 # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "f4-partnership-simulation"     # リポジトリの中の F4 フォルダ（ルートが F4 なら ""）

IN_COLAB = "google.colab" in sys.modules


def is_f4_root(p):
    return (Path(p) / "household_model.py").exists() and (Path(p) / "relationship_dynamics.py").exists()


def find_f4_root():
    for p in [Path.cwd(), Path.cwd().parent]:          # ローカル：notebooks/ の中、または F4 のルート
        if is_f4_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    if REPO_URL:                                        # Colab 方法A
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_f4_root(cand):
            return cand
    zips = glob.glob("/content/*.zip")                  # Colab 方法B
    if not zips:
        from google.colab import files
        print("F4 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/f4_upload")
    for hit in glob.glob("/content/f4_upload/**/household_model.py", recursive=True):
        if is_f4_root(Path(hit).parent):
            return Path(hit).parent
    return None


F4_ROOT = find_f4_root()
if F4_ROOT is None:
    raise RuntimeError("F4 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(F4_ROOT)
if str(F4_ROOT) not in sys.path:
    sys.path.insert(0, str(F4_ROOT))
if not (F4_ROOT / "textbook" / "max.py").exists():
    print("⚠️ textbook/ がありません。decision_strategies.py などの import が失敗します。")

# ---- 既存の .py（計算の本体）と、グラフ ----
import matplotlib.pyplot as plt
import partners as pt
import household_model as hm
import decision_strategies as ds
import relationship_dynamics as rd
import evaluation as ev
import partnership_charts as pc

pc.setup_style()                       # 文字を大きくするなら pc.setup_style(1.3)
scenarios = pt.load_scenarios()        # シナリオ A〜D（すべて架空）
params = rd.load_parameters()          # モデルのパラメータ（すべて架空）
events = pt.load_stress_events()       # ストレスイベントの組
SC = pt.scenario_by_id(scenarios, "A") # 基準にする初期条件（Scenario A）
P, HH = SC["partners"], SC["household"]
COND0 = dict(HH, stress=params["base_stress"])   # ストレスイベントのない年の生活条件
CANDS = ds.candidate_grid(params["share_step"])  # 分担の候補（全戦略で共通）
SEEDS = range(5)                       # 乱数シード
REPLICATES = 200                       # シードごとの反復回数
YEARS = params["simulation_years"]

SAVE_FIGURES = False                   # True にすると、グラフを notebooks/figures/ に PNG で保存する
FIG_DIR = "notebooks/figures"


def show(fig, name):
    """グラフを表示し、SAVE_FIGURES = True なら PNG も保存する"""
    if SAVE_FIGURES:
        print("saved:", pc.save(fig, f"{FIG_DIR}/{name}.png"))
    plt.show()


def table(rows, columns):
    """辞書のリストを表で表示する。columns = [(キー, 見出し, 書式)]。pandas があれば DataFrame で表示"""
    data = [{h: ("-" if r.get(k) is None else f.format(r[k])) for k, h, f in columns} for r in rows]
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame(data))
    except ImportError:
        print(ev.format_table(rows, columns))


print("F4_ROOT:", F4_ROOT)
print("シナリオ:", [s["label"] for s in scenarios])
print("モデルのパラメータ（架空）:", {k: params[k] for k in ("alpha", "baseline_hazard", "beta", "gamma",
                                                       "base_stress", "fairness_lambda", "share_step", "simulation_years")})

## 1. 2 人のパートナー（架空）

| 項目 | 意味 |
| --- | --- |
| `annual_income` | 年収（万円・架空） |
| `weekly_work_hours` | 1 週間の労働時間 |
| `available_housework_hours` | 1 週間に家事に使える時間の上限 |
| `preferred_housework_share` / `preferred_financial_share` | 自分が担当したい家事・生活費の割合 |
| `communication_score` | コミュニケーションの架空の指標（0〜1） |
| `fairness_weight` | 公平性を重視する程度（0〜1） |
| `satisfaction` | 関係満足度の初期値（0〜1） |

In [ ]:
table([P[pid] for pid in pt.PARTNERS],
      [("id", "partner", "{}"), ("annual_income", "income", "{}"), ("weekly_work_hours", "work h/week", "{}"),
       ("available_housework_hours", "housework limit h/week", "{}"),
       ("preferred_housework_share", "wants housework", "{:.0%}"),
       ("preferred_financial_share", "wants expenses", "{:.0%}"),
       ("communication_score", "communication", "{}"), ("fairness_weight", "fairness weight", "{}"),
       ("satisfaction", "initial satisfaction", "{}")])
print("家事の総時間:", HH["total_housework_hours"], "時間/週　生活費:", HH["annual_expenses"], "万円/年")

## 2. 家事分担

Partner A の家事分担率を $x$ とすると

$$ \text{Housework}_A = x \times \text{Total}, \qquad \text{Housework}_B = (1 - x) \times \text{Total} $$

| ルール | A の分担率 $x$ |
| --- | --- |
| 50:50 | 0.5 |
| 労働時間に応じる | B の労働時間 ÷ 2 人の労働時間（長く働く人ほど家事を少なく） |
| 所得に応じる | B の所得 ÷ 2 人の所得（**比較のためのルール。所得が多いから家事をしなくてよい、という意味ではない**） |
| 一方に集中 | 0.8（A に集中） |
| 2 人の希望 | A の希望と「B の希望から決まる A の分担」の中間 |

In [ ]:
HW_RULES = {"50:50": 0.5, "work hours": hm.work_hours_housework_share(P),
            "income": hm.income_housework_share(P), "80:20": 0.8,
            "preference": hm.preference_housework_share(P)}
hw_rows = []
for name, x in HW_RULES.items():
    a, b = hm.housework_hours(x, HH["total_housework_hours"])
    hw_rows.append({"name": name, "x": x, "A": a, "B": b,
                    "limit_A": P["A"]["available_housework_hours"], "limit_B": P["B"]["available_housework_hours"],
                    "feasible": "yes" if hm.is_feasible(x, 0.5, P, HH) else "NO"})
table(hw_rows, [("name", "rule", "{}"), ("x", "x (A's share)", "{:.2f}"), ("A", "A hours", "{:.1f}"),
                ("B", "B hours", "{:.1f}"), ("feasible", "feasible", "{}")])

In [ ]:
#@title 家事時間の棒グラフ
show(pc.plot_hours_bars(hw_rows), "01_housework_hours")

**見方**：破線は「家事に使える時間」の上限です。赤い破線は上限を超えている（実行不可能）ことを示します。

## 3. 経済的負担（生活費）

Partner A の負担率を $y$ とすると

$$ \text{Financial}_A = y \times \text{Expenses}, \qquad \text{Financial}_B = (1 - y) \times \text{Expenses} $$

**家事分担率 $x$ と生活費の負担率 $y$ は別々に決めます**（同じである必要はありません）。

In [ ]:
#@title 生活費の負担の表と棒グラフ
FIN_RULES = {"50:50": 0.5, "income-proportional": hm.income_share(P), "preference": hm.preference_financial_share(P)}
fin_rows = []
for name, y in FIN_RULES.items():
    a, b = hm.financial_amounts(y, HH["annual_expenses"])
    fin_rows.append({"name": name, "y": y, "A": a, "B": b, "A_pct": a / P["A"]["annual_income"],
                     "B_pct": b / P["B"]["annual_income"]})
table(fin_rows, [("name", "rule", "{}"), ("y", "y (A's share)", "{:.2f}"), ("A", "A pays", "{:.0f}"),
                 ("B", "B pays", "{:.0f}"), ("A_pct", "A: % of income", "{:.0%}"), ("B_pct", "B: % of income", "{:.0%}")])
show(pc.plot_money_bars(fin_rows), "01_expenses")

## 4. 所得と負担の比較

シナリオ A〜D の最初の分担で、Partner A の「所得の割合」「労働時間の割合」と「家事の割合」「生活費の割合」を並べます。
**どの分担が公平かは、何を基準にするかで変わります**（Notebook 02）。

In [ ]:
#@title 所得と負担の比較
share_rows = []
for s in scenarios:
    x, y = hm.initial_allocation(s)
    share_rows.append({"name": s["id"], "income": hm.income_share(s["partners"]),
                       "work": 1 - hm.work_hours_housework_share(s["partners"]), "housework": x, "financial": y})
show(pc.plot_share_comparison(share_rows, "Scenarios A-D: Partner A's shares"), "01_share_comparison")
for s in scenarios:
    print(f"Scenario {s['id']}: {s['description']}")

## 5. 実行可能性

家事時間が「家事に使える時間」を超える、または生活費の負担が所得を超える分担は**実行不可能**とします。
意思決定戦略（Notebook 03）は実行可能な候補だけから選びます。固定分担（Fixed）が実行不可能なときは、超過分を満足度の減点として扱います。

In [ ]:
feasible_x = [x for x in ds.share_grid(params["share_step"]) if hm.is_feasible(x, 0.5, P, HH)]
print("生活費 50:50 のとき、実行可能な家事分担率 x:", feasible_x)
for x in (0.2, 0.8):
    print(f"x = {x}:", hm.feasibility_report(x, 0.5, P, HH) or "実行可能")

## 6. 学生向け TODO：分担率を変える

In [ ]:
# TODO 1：家事分担率 x と生活費の負担率 y を変えて実行しよう
x, y = 0.6, 0.4
print("家事時間 A / B:", [round(h, 1) for h in hm.housework_hours(x, HH["total_housework_hours"])])
print("生活費   A / B:", [round(f) for f in hm.financial_amounts(y, HH["annual_expenses"])])
print("実行可能性:", hm.feasibility_report(x, y, P, HH) or "実行可能")

# TODO 2：家事の総時間を 40 時間にすると、実行可能な x の範囲はどう変わるか
hh2 = dict(HH, total_housework_hours=40)
print("総時間 40 時間で実行可能な x:", [x for x in ds.share_grid(0.1) if hm.is_feasible(x, 0.5, P, hh2)])

# TODO 3：Partner B の所得が下がり、生活費の負担が所得を超えるのはどんな条件か調べよう

## 考察（チームで記入）

- **TODO**：家事の割合と生活費の割合を、所得や労働時間の割合と比べて何が言えるか
- **TODO**：「実行可能」だが本人にとって負担が大きい分担はあるか。それはこのモデルで表せているか
- **TODO**：このモデルで表せない家事・お金の事情は何か（モデルの限界）

## 参考文献

- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第2章 最大値、第3章 線形探索、第6章 単純挿入ソート）
- F4 リポジトリの `README.md`（モデルの仮定・計算量・モデルの限界と倫理的配慮）
- データ：`data/sample_partnerships.json`（この教材のための**架空のデータ**）